# Workdo API 自動打卡系統

此 Notebook 提供 Workdo API 自動上下班打卡功能。

## 設定步驟

1. 建立 `.env` 檔案在專案根目錄
2. 填入以下環境變數：
   ```
   WORKDO_EMAIL=your_email@example.com
   WORKDO_PASSWORD=your_password
   WORKDO_COMPANY_ID=your_company_id
   WORKDO_API_URL=https://api.workdo.co
   ```

3. 設定打卡時間（預設為上班 09:00，下班 18:00）

In [ ]:
import os
import requests
import schedule
import time
from datetime import datetime
from dotenv import load_dotenv

# 載入環境變數
load_dotenv()

print("✅ 套件導入完成")

In [ ]:
class WorkdoAPI:
    """Workdo API 自動打卡類別"""
    
    def __init__(self):
        self.email = os.getenv('WORKDO_EMAIL')
        self.password = os.getenv('WORKDO_PASSWORD')
        self.company_id = os.getenv('WORKDO_COMPANY_ID')
        self.api_url = os.getenv('WORKDO_API_URL', 'https://api.workdo.co')
        self.token = None
        self.session = requests.Session()
        
    def login(self):
        """登入 Workdo 取得存取權杖"""
        try:
            url = f"{self.api_url}/api/v1/auth/login"
            payload = {
                'email': self.email,
                'password': self.password,
                'company_id': self.company_id
            }
            response = self.session.post(url, json=payload)
            response.raise_for_status()
            
            data = response.json()
            self.token = data.get('token')
            self.session.headers.update({'Authorization': f'Bearer {self.token}'})
            
            print(f"✅ 登入成功 - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            return True
            
        except requests.exceptions.RequestException as e:
            print(f"❌ 登入失敗: {str(e)}")
            return False
    
    def clock_in(self):
        """上班打卡"""
        try:
            url = f"{self.api_url}/api/v1/attendance/clock-in"
            payload = {
                'timestamp': datetime.now().isoformat(),
                'type': 'clock_in'
            }
            response = self.session.post(url, json=payload)
            response.raise_for_status()
            
            print(f"✅ 上班打卡成功 - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            return True
            
        except requests.exceptions.RequestException as e:
            print(f"❌ 上班打卡失敗: {str(e)}")
            return False
    
    def clock_out(self):
        """下班打卡"""
        try:
            url = f"{self.api_url}/api/v1/attendance/clock-out"
            payload = {
                'timestamp': datetime.now().isoformat(),
                'type': 'clock_out'
            }
            response = self.session.post(url, json=payload)
            response.raise_for_status()
            
            print(f"✅ 下班打卡成功 - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            return True
            
        except requests.exceptions.RequestException as e:
            print(f"❌ 下班打卡失敗: {str(e)}")
            return False
    
    def get_attendance_status(self):
        """查詢今日打卡狀態"""
        try:
            url = f"{self.api_url}/api/v1/attendance/status"
            response = self.session.get(url)
            response.raise_for_status()
            
            data = response.json()
            print(f"📊 今日打卡狀態: {data}")
            return data
            
        except requests.exceptions.RequestException as e:
            print(f"❌ 查詢狀態失敗: {str(e)}")
            return None

# 建立 API 實例
workdo = WorkdoAPI()
print("✅ Workdo API 類別建立完成")

In [ ]:
class AutoClockScheduler:
    """自動打卡排程器"""
    
    def __init__(self, workdo_api, clock_in_time="09:00", clock_out_time="18:00"):
        self.workdo = workdo_api
        self.clock_in_time = clock_in_time
        self.clock_out_time = clock_out_time
        self.is_running = False
        
    def job_clock_in(self):
        """上班打卡任務"""
        print(f"\n🔔 執行上班打卡任務...")
        if not self.workdo.token:
            self.workdo.login()
        self.workdo.clock_in()
    
    def job_clock_out(self):
        """下班打卡任務"""
        print(f"\n🔔 執行下班打卡任務...")
        if not self.workdo.token:
            self.workdo.login()
        self.workdo.clock_out()
    
    def setup_schedule(self):
        """設定排程"""
        schedule.clear()
        schedule.every().day.at(self.clock_in_time).do(self.job_clock_in)
        schedule.every().day.at(self.clock_out_time).do(self.job_clock_out)
        
        print(f"⏰ 排程設定完成:")
        print(f"   - 上班打卡時間: {self.clock_in_time}")
        print(f"   - 下班打卡時間: {self.clock_out_time}")
    
    def run(self, duration_minutes=None):
        """
        啟動排程器
        
        Args:
            duration_minutes: 執行時長（分鐘），None 表示持續執行
        """
        self.setup_schedule()
        self.is_running = True
        
        print(f"\n🚀 自動打卡排程器已啟動")
        print(f"   執行時長: {'持續執行' if duration_minutes is None else f'{duration_minutes} 分鐘'}")
        print(f"   當前時間: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"\n💡 提示: 在 Jupyter Notebook 中按 '■' 停止執行")
        
        start_time = time.time()
        
        try:
            while self.is_running:
                schedule.run_pending()
                time.sleep(60)  # 每分鐘檢查一次
                
                # 如果設定了執行時長，檢查是否超時
                if duration_minutes and (time.time() - start_time) > duration_minutes * 60:
                    print(f"\n⏰ 達到設定執行時長 ({duration_minutes} 分鐘)，停止排程器")
                    break
                    
        except KeyboardInterrupt:
            print(f"\n⏹️ 排程器已停止")
            self.is_running = False
    
    def stop(self):
        """停止排程器"""
        self.is_running = False
        schedule.clear()
        print("⏹️ 排程器已停止")

print("✅ 自動打卡排程器類別建立完成")

## 使用方式

### 方式 1: 手動打卡
直接呼叫打卡方法進行測試

In [ ]:
# 測試登入
if workdo.login():
    print("登入測試成功，可以開始使用打卡功能")
else:
    print("請檢查 .env 檔案設定")

In [ ]:
# 手動上班打卡
workdo.clock_in()

In [ ]:
# 手動下班打卡
workdo.clock_out()

In [ ]:
# 查詢今日打卡狀態
workdo.get_attendance_status()

### 方式 2: 自動排程打卡
設定每日自動上下班打卡時間

In [ ]:
# 建立自動排程器（設定上班 09:00，下班 18:00）
scheduler = AutoClockScheduler(
    workdo_api=workdo,
    clock_in_time="09:00",
    clock_out_time="18:00"
)

# 啟動排程器（測試模式：執行 10 分鐘）
# 注意：實際使用時，移除 duration_minutes 參數以持續執行
# scheduler.run(duration_minutes=10)

print("✅ 排程器已建立")
print("💡 執行 scheduler.run() 即可啟動自動打卡")
print("💡 執行 scheduler.run(duration_minutes=10) 可設定執行 10 分鐘後停止")

### 方式 3: 立即測試排程（不等待設定時間）
可以手動觸發排程任務進行測試

In [ ]:
# 立即執行上班打卡任務（用於測試）
scheduler.job_clock_in()

In [ ]:
# 立即執行下班打卡任務（用於測試）
scheduler.job_clock_out()

## 注意事項

1. **環境變數設定**: 請確保已建立 `.env` 檔案並正確填寫 Workdo 帳號資料
2. **API URL**: 請依照實際的 Workdo API 網址修改 `WORKDO_API_URL`
3. **時區設定**: 確保系統時間正確，排程會依據系統時間執行
4. **持續執行**: 在生產環境中，建議使用系統排程工具（如 cron）或部署到雲端服務
5. **錯誤處理**: 如遇到登入或打卡失敗，請檢查網路連線和 API 認證資訊
6. **測試建議**: 先使用手動打卡方式測試 API 連線，確認無誤後再啟用自動排程

## 進階配置

如需在伺服器上持續執行，建議：
- 使用 Docker 容器化部署
- 使用 systemd 服務管理
- 使用雲端服務（如 AWS Lambda、Google Cloud Functions）定時觸發
- 加入日誌記錄功能，追蹤打卡歷史